# Loading

In [ ]:
import scanpy as sc
import anndata as ad
import numpy as np
import pandas as pd
import scipy.sparse as sp
import time
import os

from spagrn.regulatory_network import InferNetwork as irn
from _3D_plot import plot_gene_3d_k3d

In [ ]:
adata_grn = sc.read_h5ad("Output/GRN_GPU_output/midbrain/grn_midbrain_spagrn.h5ad")
adata_grn

In [ ]:
grn = adata_grn.uns['regulon_dict']
del adata_grn
grn.keys()

# Cellbin data process

In [ ]:
import pandas as pd
import scanpy as sc
from pathlib import Path
import sys

# --- 1. Define Paths ---

# Input directory (containing .h5ad files)
data_dir = Path("fetal_brain_hongtaoyu/3dCellBin/NB20241129094951824")

# Find all .h5ad files
h5ad_files = list(data_dir.glob("*.h5ad"))

# Output base directory
output_base_dir = Path("Process_Data/cellbin_3D_region")

# The specific region to extract
region_to_extract = 'midbrain'
# The column name in the .h5ad's .obs to check
column_to_check = 'regions' 

# --- 2. Create Output Directory ---
# We will save all 'midbrain' files into a subfolder of the same name
region_output_dir = output_base_dir / region_to_extract
region_output_dir.mkdir(parents=True, exist_ok=True)
print(f"✅ Output directory set: {region_output_dir}")

# Check if any files were found
if not h5ad_files:
    print(f"❌ [Error] No .h5ad files found in {data_dir}. Please check the path.")
    sys.exit() # Exit the script if no files are found

print(f"ℹ️ Found {len(h5ad_files)} .h5ad files. Starting processing...")
print("-" * 30)

# --- 3. Loop through each file ---
for h5ad_file_path in h5ad_files:
    file_name = h5ad_file_path.name  # e.g., "sample_A.h5ad"
    
    print(f"\n>>> Processing file: {file_name}")

    try:
        # --- 3.1 Read h5ad file directly ---
        # (All CSV reading and merging logic has been removed)
        adata = sc.read_h5ad(h5ad_file_path)
            
        # --- 3.2 Check for 'regions' column in the .h5ad's .obs ---
        if column_to_check not in adata.obs.columns:
            print(f"  [Warning] Column '{column_to_check}' not found in {file_name}'s .obs. Skipping this file.")
            continue

        # --- 3.3 Check for 'midbrain' and save ---
        
        # 1. Check if 'midbrain' exists in the 'regions' values for this file
        if region_to_extract not in adata.obs[column_to_check].values:
            print(f"  [Note] Region '{region_to_extract}' not found in {file_name}. Skipping this file.")
            continue
        
        # 2. If found, filter the data
        print(f"  ✅ Found '{region_to_extract}'. Filtering and saving...")
        adata_region_subset = adata[adata.obs[column_to_check] == region_to_extract].copy()
        
        if adata_region_subset.n_obs == 0:
            print(f"  [Warning] Region '{region_to_extract}' contains 0 spots. Not saving empty file.")
            continue
            
        # 3. Define the final output file path
        output_h5ad_path = region_output_dir / file_name
        
        # 4. Save the subsetted h5ad file
        adata_region_subset.write_h5ad(output_h5ad_path)
        print(f"  💾 Saved: {output_h5ad_path} (Contains {adata_region_subset.n_obs} spots)")
             
        print(f"  <<< Finished processing file: {file_name}")

    except Exception as e:
        print(f"  [!! CRITICAL ERROR !!] An unexpected error occurred while processing {file_name}: {e}")
        continue

print("-" * 30)
print("All files processed.")

In [ ]:
base_dir = Path("Process_Data/cellbin_3D_region")
for region_dir in [p for p in base_dir.iterdir() if p.is_dir()]:
    region_name = region_dir.name
    print(f"\n>>> Processing region: {region_name}")

    try:
        files_to_merge = list(region_dir.glob("*.h5ad"))
        
        adata_list = [sc.read_h5ad(f) for f in files_to_merge]
        keys = [f.stem for f in files_to_merge] # Use filename as key

        combined_adata = ad.concat(
            adata_list,
            label='sample_id',  # Creates a new .obs column
            keys=keys,          # Uses filenames to fill 'sample_id'
            index_unique='-',   # Ensures unique obs_names (e.g., "filename-barcode")
            join='outer'        # Keeps all genes (union)
        )

        output_path = base_dir / f"combined_adata_{region_name}.h5ad"
        print(f"  Saving merged file to: {output_path}")
        combined_adata.write_h5ad(output_path, compression='gzip')
        
        print(f"  ✅ Region '{region_name}' complete (Total spots: {combined_adata.n_obs})")

    except Exception as e:
        continue # Skip to the next region

print("\n" + "=" * 40)
print("🎉 All processing complete.")

In [ ]:
adata_merge = sc.read_h5ad("Process_Data/cellbin_3D_region/combined_adata_midbrain.h5ad")
adata_merge